# SafeDoge — Dog Risk Classification Model

Multi-task deep learning model that classifies **breed** (120 classes) and **emotion** (4 classes) from dog photos, then computes a **risk score** with full breakdown.

**Architecture:** EfficientNet-B0 backbone → shared features → breed head + emotion head

**Datasets:**
- Stanford Dogs Dataset (120 breeds, ~20k images)
- Dog Emotion Dataset (angry, happy, sad, relaxed)

**Outputs:**
- `dog_risk_model.pth` — PyTorch weights
- `dog_risk_model.onnx` — ONNX export
- `breed_labels.json`, `emotion_labels.json`, `breed_risk_table.json`

---

## 1. Setup & Dependencies

In [ ]:
!pip install efficientnet-pytorch tqdm scikit-learn matplotlib seaborn -q

In [ ]:
import os, json, time, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from efficientnet_pytorch import EfficientNet

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Configuration
Config = {
    'IMG_SIZE': 224,
    'BATCH_SIZE': 32,
    'NUM_BREEDS': 120,
    'NUM_EMOTIONS': 4,
    'EPOCHS': 25,
    'LR': 1e-4,
    'WEIGHT_DECAY': 1e-4,
    'BREED_LOSS_WEIGHT': 0.6,
    'EMOTION_LOSS_WEIGHT': 0.4,
    'EARLY_STOP_PATIENCE': 7,
    'SEED': 42,
    'NUM_WORKERS': 2,
    'TRAIN_SPLIT': 0.8,
    'VAL_SPLIT': 0.1,
    'TEST_SPLIT': 0.1,
}

torch.manual_seed(Config['SEED'])
np.random.seed(Config['SEED'])

WORK_DIR = Path('/kaggle/working')
DATA_DIR = Path('/kaggle/data')
MODEL_DIR = WORK_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

print(f"Config: {json.dumps(Config, indent=2)}")

---
## 2. Download & Organize Datasets

In [ ]:
# Download Stanford Dogs Dataset (120 breeds)
!kaggle datasets download -d jessicali9530/stanford-dogs-dataset -p /kaggle/data/stanford_dogs --unzip -q

# Download Dog Emotion Dataset (4 classes)
!kaggle datasets download -d mohitagarwal17/dog-emotion-datasetcleaned-version -p /kaggle/data/dog_emotion --unzip -q

print("Datasets downloaded.")

In [ ]:
# Explore Stanford Dogs structure
stanford_base = Path('/kaggle/data/stanford_dogs')
print("Stanford Dogs contents:")
for p in sorted(stanford_base.iterdir()):
    print(f"  {p.name}/")
    if p.is_dir():
        subdirs = list(p.iterdir())[:5]
        for s in subdirs:
            if s.is_dir():
                count = len(list(s.glob('*.jpg')))
                print(f"    {s.name}: {count} images")
        print(f"    ... ({len(subdirs)} shown)")

In [ ]:
# Explore Dog Emotion structure
emotion_base = Path('/kaggle/data/dog_emotion')
print("Dog Emotion contents:")
for p in sorted(emotion_base.iterdir()):
    print(f"  {p.name}/")
    if p.is_dir():
        for s in sorted(p.iterdir())[:10]:
            if s.is_dir():
                count = len(list(s.glob('*.jpg')) + list(s.glob('*.png')))
                print(f"    {s.name}: {count} images")

In [ ]:
# Organize both datasets into a unified structure
# We create a combined dataset where each image has both breed and emotion labels
# For images that only have one label, we use 'unknown' for the other

import shutil

COMBINED_DIR = WORK_DIR / 'dataset'
BREED_DIR = COMBINED_DIR / 'breeds'
EMOTION_DIR = COMBINED_DIR / 'emotions'

BREED_DIR.mkdir(parents=True, exist_ok=True)
EMOTION_DIR.mkdir(parents=True, exist_ok=True)

# --- Organize Stanford Dogs into breeds ---
# The dataset has Images/ folder with breed-named subfolders
stanford_images = stanford_base / 'Images'
if not stanford_images.exists():
    # Try alternate structure
    for p in stanford_base.rglob('Images'):
        if p.is_dir():
            stanford_images = p
            break

breed_classes = []
breed_counts = {}
image_breed_map = {}  # filename -> breed

for breed_folder in sorted(stanford_images.iterdir()):
    if not breed_folder.is_dir():
        continue
    # Clean breed name: n02085620-Chihuahua -> Chihuahua
    breed_name = breed_folder.name.split('-', 1)[-1].replace('_', ' ') if '-' in breed_folder.name else breed_folder.name
    breed_classes.append(breed_name)
    
    dest = BREED_DIR / breed_name
    dest.mkdir(exist_ok=True)
    
    count = 0
    for img_file in breed_folder.glob('*.jpg'):
        new_name = f"{breed_name}_{count:04d}.jpg"
        shutil.copy2(img_file, dest / new_name)
        image_breed_map[new_name] = breed_name
        count += 1
    breed_counts[breed_name] = count

print(f"Organized {len(breed_classes)} breeds, {sum(breed_counts.values())} total images")
print(f"Top 10 breeds by count:")
for breed, count in sorted(breed_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {breed}: {count}")

In [ ]:
# --- Organize Dog Emotion into emotions ---
# Find the emotion images directory
emotion_images = None
for p in emotion_base.rglob('*'):
    if p.is_dir() and p.name in ['angry', 'happy', 'sad', 'relaxed', 'Angry', 'Happy', 'Sad', 'Relaxed']:
        emotion_images = p.parent
        break

if emotion_images is None:
    # Fallback: look for subdirectories containing images
    for p in emotion_base.rglob('*'):
        if p.is_dir() and any(p.glob('*.jpg')):
            emotion_images = p.parent
            break

emotion_classes = ['angry', 'happy', 'sad', 'relaxed']
emotion_counts = {}
image_emotion_map = {}

if emotion_images:
    for emotion_folder in sorted(emotion_images.iterdir()):
        if not emotion_folder.is_dir():
            continue
        emotion_name = emotion_folder.name.lower()
        if emotion_name not in emotion_classes:
            continue
        
        dest = EMOTION_DIR / emotion_name
        dest.mkdir(exist_ok=True)
        
        count = 0
        for img_file in list(emotion_folder.glob('*.jpg')) + list(emotion_folder.glob('*.png')):
            new_name = f"{emotion_name}_{count:04d}.jpg"
            shutil.copy2(img_file, dest / new_name)
            image_emotion_map[new_name] = emotion_name
            count += 1
        emotion_counts[emotion_name] = count

print(f"Organized emotions: {emotion_counts}")

---
## 3. Exploratory Data Analysis

In [ ]:
# Breed distribution
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Breed distribution (top 30)
top_breeds = sorted(breed_counts.items(), key=lambda x: -x[1])[:30]
breed_names = [b[0] for b in top_breeds]
breed_vals = [b[1] for b in top_breeds]
axes[0].barh(range(len(breed_names)), breed_vals, color='#2563EB')
axes[0].set_yticks(range(len(breed_names)))
axes[0].set_yticklabels(breed_names, fontsize=8)
axes[0].set_xlabel('Number of Images')
axes[0].set_title('Top 30 Breeds by Image Count')
axes[0].invert_yaxis()

# Emotion distribution
emo_names = list(emotion_counts.keys())
emo_vals = list(emotion_counts.values())
colors = ['#EF4444', '#22C55E', '#F59E0B', '#6366F1']
axes[1].bar(emo_names, emo_vals, color=colors)
axes[1].set_ylabel('Number of Images')
axes[1].set_title('Emotion Class Distribution')
for i, v in enumerate(emo_vals):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(MODEL_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Total breeds: {len(breed_classes)}")
print(f"Total breed images: {sum(breed_counts.values())}")
print(f"Total emotion images: {sum(emotion_counts.values())}")

In [ ]:
# Sample images from each emotion class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, emotion in enumerate(emotion_classes):
    emotion_dir = EMOTION_DIR / emotion
    if not emotion_dir.exists():
        continue
    samples = list(emotion_dir.glob('*.jpg'))[:4]
    for j, img_path in enumerate(samples):
        ax = axes[i % 2, j]
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(emotion.capitalize() if j == 0 else '', fontsize=12, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Images per Emotion Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'sample_emotions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample images from breeds
sample_breeds = list(breed_counts.keys())[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, breed in enumerate(sample_breeds):
    breed_dir = BREED_DIR / breed
    samples = list(breed_dir.glob('*.jpg'))[:1]
    if samples:
        img = Image.open(samples[0])
        ax = axes[i // 4, i % 4]
        ax.imshow(img)
        ax.set_title(breed, fontsize=9, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Images per Breed', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'sample_breeds.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Dataset & Data Loaders

In [ ]:
# Build unified dataset catalog
# Each image gets a breed label and an emotion label
# Images from Stanford Dogs get breed=actual, emotion=unknown sentinel
# Images from Dog Emotion get breed=unknown sentinel, emotion=actual

BREED_TO_IDX = {b: i for i, b in enumerate(sorted(breed_classes))}
EMOTION_TO_IDX = {e: i for i, e in enumerate(emotion_classes)}
IDX_TO_BREED = {i: b for b, i in BREED_TO_IDX.items()}
IDX_TO_EMOTION = {i: e for e, i in EMOTION_TO_IDX.items()}

NUM_BREEDS = len(BREED_TO_IDX)
NUM_EMOTIONS = len(EMOTION_TO_IDX)
# Sentinel indices for samples missing the other label (masked in loss, outside valid class range)
BREED_UNKNOWN_IDX = NUM_BREEDS
EMOTION_UNKNOWN_IDX = NUM_EMOTIONS

print(f"Breed classes: {NUM_BREEDS}")
print(f"Emotion classes: {NUM_EMOTIONS}")
print(f"Emotion mapping: {EMOTION_TO_IDX}")


In [ ]:
class DogRiskDataset(Dataset):
    """Multi-task dataset for breed + emotion classification."""
    
    def __init__(self, image_paths, breed_labels, emotion_labels, transform=None):
        self.image_paths = image_paths
        self.breed_labels = breed_labels
        self.emotion_labels = emotion_labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        breed = self.breed_labels[idx]
        emotion = self.emotion_labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(breed, dtype=torch.long), torch.tensor(emotion, dtype=torch.long)

In [ ]:
# Data augmentation & normalization (ImageNet stats)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(Config['IMG_SIZE']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(Config['IMG_SIZE']),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("Transforms defined.")

In [ ]:
# Build full catalog: combine breed images + emotion images
all_image_paths = []
all_breed_labels = []
all_emotion_labels = []

# Add breed images (emotion = 'unknown')
unknown_emotion_idx = EMOTION_UNKNOWN_IDX
for breed_name in breed_classes:
    breed_dir = BREED_DIR / breed_name
    if not breed_dir.exists():
        continue
    for img_path in sorted(breed_dir.glob('*.jpg')):
        all_image_paths.append(str(img_path))
        all_breed_labels.append(BREED_TO_IDX[breed_name])
        all_emotion_labels.append(unknown_emotion_idx)  # unknown emotion

# Add emotion images (breed = 'unknown')
unknown_breed_idx = BREED_UNKNOWN_IDX
for emotion_name in emotion_classes:
    emotion_dir = EMOTION_DIR / emotion_name
    if not emotion_dir.exists():
        continue
    for img_path in sorted(emotion_dir.glob('*.jpg')):
        all_image_paths.append(str(img_path))
        all_breed_labels.append(unknown_breed_idx)
        all_emotion_labels.append(EMOTION_TO_IDX[emotion_name])

print(f"Total images: {len(all_image_paths)}")
print(f"  Breed-labeled: {sum(1 for b in all_breed_labels if b != unknown_breed_idx)}")
print(f"  Emotion-labeled: {sum(1 for e in all_emotion_labels if e != unknown_emotion_idx)}")
print(f"  Both: {sum(1 for b, e in zip(all_breed_labels, all_emotion_labels) if b != unknown_breed_idx and e != unknown_emotion_idx)}")

In [ ]:
# Train/Val/Test split (stratified by breed)
from sklearn.model_selection import train_test_split

indices = list(range(len(all_image_paths)))

# First split: train+val vs test
train_val_idx, test_idx = train_test_split(
    indices,
    test_size=Config['TEST_SPLIT'],
    random_state=Config['SEED'],
    stratify=all_breed_labels,
)

# Second split: train vs val
relative_val_size = Config['VAL_SPLIT'] / (Config['TRAIN_SPLIT'] + Config['VAL_SPLIT'])
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=relative_val_size,
    random_state=Config['SEED'],
    stratify=[all_breed_labels[i] for i in train_val_idx],
)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

# Create datasets
train_dataset = DogRiskDataset(
    [all_image_paths[i] for i in train_idx],
    [all_breed_labels[i] for i in train_idx],
    [all_emotion_labels[i] for i in train_idx],
    transform=train_transform,
)

val_dataset = DogRiskDataset(
    [all_image_paths[i] for i in val_idx],
    [all_breed_labels[i] for i in val_idx],
    [all_emotion_labels[i] for i in val_idx],
    transform=val_test_transform,
)

test_dataset = DogRiskDataset(
    [all_image_paths[i] for i in test_idx],
    [all_breed_labels[i] for i in test_idx],
    [all_emotion_labels[i] for i in test_idx],
    transform=val_test_transform,
)

# Data loaders
train_loader = DataLoader(
    train_dataset, batch_size=Config['BATCH_SIZE'], shuffle=True,
    num_workers=Config['NUM_WORKERS'], pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=Config['BATCH_SIZE'], shuffle=False,
    num_workers=Config['NUM_WORKERS'], pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=Config['BATCH_SIZE'], shuffle=False,
    num_workers=Config['NUM_WORKERS'], pin_memory=True,
)

print(f"DataLoaders ready. Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

In [ ]:
# Verify a batch
images, breeds, emotions = next(iter(train_loader))
print(f"Batch shape: {images.shape}")  # [B, 3, 224, 224]
print(f"Breed labels: {breeds[:5]}")
print(f"Emotion labels: {emotions[:5]}")

# Denormalize and show samples
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(8):
    img = images[i].numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img = np.clip(img, 0, 1)
    ax = axes[i // 4, i % 4]
    ax.imshow(img)
    ax.set_title(f"Breed: {IDX_TO_BREED[breeds[i].item()][:15]}\nEmotion: {IDX_TO_EMOTION[emotions[i].item()]}", fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Training Batch', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Model Definition

In [ ]:
class MultiTaskDogModel(nn.Module):
    """
    Multi-task model: EfficientNet-B0 backbone with breed + emotion heads.
    
    Architecture:
        Input (3, 224, 224)
        -> EfficientNet-B0 backbone (pretrained on ImageNet)
        -> Global Average Pooling -> 1280-d feature vector
        -> Breed Head: Dropout(0.3) -> Linear(1280, 512) -> ReLU -> Dropout(0.2) -> Linear(512, num_breeds)
        -> Emotion Head: Dropout(0.3) -> Linear(1280, 128) -> ReLU -> Linear(128, num_emotions)
    """
    
    def __init__(self, num_breeds=NUM_BREEDS, num_emotions=NUM_EMOTIONS):
        super().__init__()
        
        # Backbone: EfficientNet-B0 (ImageNet pretrained)
        self.backbone = EfficientNet.from_pretrained('efficientnet-b0')
        backbone_out_features = 1280  # EfficientNet-B0 output features
        
        # Remove the original classifier
        self.backbone._fc = nn.Identity()
        
        # Breed classification head (120 classes)
        self.breed_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(backbone_out_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_breeds),
        )
        
        # Emotion classification head (4 classes)
        self.emotion_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(backbone_out_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_emotions),
        )
    
    def forward(self, x):
        features = self.backbone(x)  # [B, 1280]
        breed_logits = self.breed_head(features)    # [B, num_breeds]
        emotion_logits = self.emotion_head(features) # [B, num_emotions]
        return breed_logits, emotion_logits
    
    def predict(self, x):
        """Returns predicted class indices and probabilities."""
        self.eval()
        with torch.no_grad():
            breed_logits, emotion_logits = self.forward(x)
            breed_probs = F.softmax(breed_logits, dim=-1)
            emotion_probs = F.softmax(emotion_logits, dim=-1)
            breed_pred = torch.argmax(breed_probs, dim=-1)
            emotion_pred = torch.argmax(emotion_probs, dim=-1)
        return breed_pred, emotion_pred, breed_probs, emotion_probs

In [ ]:
# Initialize model
model = MultiTaskDogModel(num_breeds=NUM_BREEDS, num_emotions=NUM_EMOTIONS)
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.1f} MB (float32)")

# Test forward pass
dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
breed_out, emotion_out = model(dummy)
print(f"\nForward pass test:")
print(f"  Breed output: {breed_out.shape}  (expected [2, {NUM_BREEDS}])")
print(f"  Emotion output: {emotion_out.shape}  (expected [2, {NUM_EMOTIONS}])")

---
## 6. Training Loop

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


def train_one_epoch(model, loader, optimizer, scheduler, device, breed_weight, emotion_weight):
    model.train()
    total_loss = 0
    breed_correct = 0
    emotion_correct = 0
    total = 0
    
    for images, breeds, emotions in loader:
        images = images.to(device, non_blocking=True)
        breeds = breeds.to(device, non_blocking=True)
        emotions = emotions.to(device, non_blocking=True)
        
        # Filter out 'unknown' labels
        breed_mask = breeds != BREED_UNKNOWN_IDX
        emotion_mask = emotions != EMOTION_UNKNOWN_IDX
        
        optimizer.zero_grad()
        breed_logits, emotion_logits = model(images)
        
        # Multi-task loss
        loss = torch.tensor(0.0, device=device)
        
        if breed_mask.sum() > 0:
            breed_loss = F.cross_entropy(breed_logits[breed_mask], breeds[breed_mask])
            loss += breed_weight * breed_loss
            breed_pred = breed_logits[breed_mask].argmax(dim=-1)
            breed_correct += (breed_pred == breeds[breed_mask]).sum().item()
        
        if emotion_mask.sum() > 0:
            emotion_loss = F.cross_entropy(emotion_logits[emotion_mask], emotions[emotion_mask])
            loss += emotion_weight * emotion_loss
            emotion_pred = emotion_logits[emotion_mask].argmax(dim=-1)
            emotion_correct += (emotion_pred == emotions[emotion_mask]).sum().item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        total += images.size(0)
    
    scheduler.step()
    
    return {
        'loss': total_loss / total,
        'breed_acc': breed_correct / max(1, sum(1 for b in train_dataset.breed_labels if b != BREED_UNKNOWN_IDX)) * 100,
        'emotion_acc': emotion_correct / max(1, sum(1 for e in train_dataset.emotion_labels if e != EMOTION_UNKNOWN_IDX)) * 100,
    }


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_loss = 0
    breed_correct = 0
    emotion_correct = 0
    breed_total = 0
    emotion_total = 0
    total = 0
    
    for images, breeds, emotions in loader:
        images = images.to(device, non_blocking=True)
        breeds = breeds.to(device, non_blocking=True)
        emotions = emotions.to(device, non_blocking=True)
        
        breed_mask = breeds != BREED_UNKNOWN_IDX
        emotion_mask = emotions != EMOTION_UNKNOWN_IDX
        
        breed_logits, emotion_logits = model(images)
        
        loss = torch.tensor(0.0, device=device)
        
        if breed_mask.sum() > 0:
            loss += Config['BREED_LOSS_WEIGHT'] * F.cross_entropy(breed_logits[breed_mask], breeds[breed_mask])
            breed_pred = breed_logits[breed_mask].argmax(dim=-1)
            breed_correct += (breed_pred == breeds[breed_mask]).sum().item()
            breed_total += breed_mask.sum().item()
        
        if emotion_mask.sum() > 0:
            loss += Config['EMOTION_LOSS_WEIGHT'] * F.cross_entropy(emotion_logits[emotion_mask], emotions[emotion_mask])
            emotion_pred = emotion_logits[emotion_mask].argmax(dim=-1)
            emotion_correct += (emotion_pred == emotions[emotion_mask]).sum().item()
            emotion_total += emotion_mask.sum().item()
        
        total_loss += loss.item() * images.size(0)
        total += images.size(0)
    
    return {
        'loss': total_loss / total,
        'breed_acc': breed_correct / max(1, breed_total) * 100,
        'emotion_acc': emotion_correct / max(1, emotion_total) * 100,
    }

print("Training functions defined.")

In [ ]:
# Setup optimizer, scheduler, early stopping
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=Config['LR'],
    weight_decay=Config['WEIGHT_DECAY'],
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=Config['EPOCHS'],
    eta_min=1e-6,
)

early_stopping = EarlyStopping(patience=Config['EARLY_STOP_PATIENCE'])

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

print(f"Optimizer: Adam (lr={Config['LR']}, wd={Config['WEIGHT_DECAY']})")
print(f"Scheduler: CosineAnnealing (T_max={Config['EPOCHS']})")
print(f"Mixed precision: {'Yes' if scaler else 'No'}")

In [ ]:
# Training loop
history = {'train_loss': [], 'val_loss': [], 'train_breed_acc': [], 'val_breed_acc': [],
           'train_emotion_acc': [], 'val_emotion_acc': [], 'lr': []}

best_val_loss = float('inf')
best_model_state = None

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Breed Acc':>10} | {'Emotion Acc':>12} | {'LR':>10} | {'Time':>6}")
print("-" * 85)

for epoch in range(Config['EPOCHS']):
    start = time.time()
    
    # Train
    if scaler:
        # Mixed precision training
        model.train()
        epoch_loss = 0
        for images, breeds, emotions in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            breeds = breeds.to(DEVICE, non_blocking=True)
            emotions = emotions.to(DEVICE, non_blocking=True)
            
            breed_mask = breeds != BREED_UNKNOWN_IDX
            emotion_mask = emotions != EMOTION_UNKNOWN_IDX
            
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                breed_logits, emotion_logits = model(images)
                loss = torch.tensor(0.0, device=DEVICE)
                if breed_mask.sum() > 0:
                    loss += Config['BREED_LOSS_WEIGHT'] * F.cross_entropy(breed_logits[breed_mask], breeds[breed_mask])
                if emotion_mask.sum() > 0:
                    loss += Config['EMOTION_LOSS_WEIGHT'] * F.cross_entropy(emotion_logits[emotion_mask], emotions[emotion_mask])
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item() * images.size(0)
        
        scheduler.step()
        train_loss = epoch_loss / len(train_dataset)
    else:
        train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler, DEVICE,
                                        Config['BREED_LOSS_WEIGHT'], Config['EMOTION_LOSS_WEIGHT'])
        train_loss = train_metrics['loss']
    
    # Validate
    val_metrics = validate(model, val_loader, DEVICE)
    
    elapsed = time.time() - start
    lr_now = scheduler.get_last_lr()[0]
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_breed_acc'].append(val_metrics['breed_acc'])
    history['val_emotion_acc'].append(val_metrics['emotion_acc'])
    history['lr'].append(lr_now)
    
    print(f"{epoch+1:5d} | {train_loss:10.4f} | {val_metrics['loss']:10.4f} | {val_metrics['breed_acc']:9.2f}% | {val_metrics['emotion_acc']:11.2f}% | {lr_now:10.6f} | {elapsed:5.1f}s")
    
    # Save best model
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, MODEL_DIR / 'best_model.pth')
    
    # Early stopping
    early_stopping(val_metrics['loss'])
    if early_stopping.early_stop:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Breed accuracy
axes[1].plot(history['val_breed_acc'], label='Val Breed Acc', color='#2563EB', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Validation Breed Accuracy')
axes[1].grid(True, alpha=0.3)

# Emotion accuracy
axes[2].plot(history['val_emotion_acc'], label='Val Emotion Acc', color='#22C55E', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Validation Emotion Accuracy')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pth', map_location=DEVICE))
model.eval()

# Evaluate on test set
all_breed_preds = []
all_breed_true = []
all_emotion_preds = []
all_emotion_true = []

with torch.no_grad():
    for images, breeds, emotions in test_loader:
        images = images.to(DEVICE)
        breed_logits, emotion_logits = model(images)
        
        # Only collect predictions for known labels
        for i in range(len(breeds)):
            if breeds[i] != BREED_UNKNOWN_IDX:
                all_breed_preds.append(breed_logits[i].argmax().item())
                all_breed_true.append(breeds[i].item())
            if emotions[i] != EMOTION_UNKNOWN_IDX:
                all_emotion_preds.append(emotion_logits[i].argmax().item())
                all_emotion_true.append(emotions[i].item())

print(f"Test samples with breed labels: {len(all_breed_true)}")
print(f"Test samples with emotion labels: {len(all_emotion_true)}")

In [ ]:
# Classification report for emotions
emotion_target_names = [IDX_TO_EMOTION[i] for i in range(NUM_EMOTIONS)]
print("=" * 60)
print("EMOTION CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(all_emotion_true, all_emotion_preds,
                            target_names=emotion_target_names, digits=3))

In [ ]:
# Confusion matrix for emotions
cm_emotion = confusion_matrix(all_emotion_true, all_emotion_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_emotion, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_target_names, yticklabels=emotion_target_names, ax=ax)
ax.set_xlabel('Predicted', fontweight='bold')
ax.set_ylabel('True', fontweight='bold')
ax.set_title('Emotion Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'emotion_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top-5 breed accuracy
breed_correct_top1 = 0
breed_correct_top5 = 0

with torch.no_grad():
    for images, breeds, emotions in test_loader:
        images = images.to(DEVICE)
        breed_logits, _ = model(images)
        breed_probs = F.softmax(breed_logits, dim=-1)
        
        for i in range(len(breeds)):
            if breeds[i] != BREED_UNKNOWN_IDX:
                true_idx = breeds[i].item()
                top1 = breed_probs[i].argmax().item()
                top5 = breed_probs[i].topk(5).indices.tolist()
                if top1 == true_idx:
                    breed_correct_top1 += 1
                if true_idx in top5:
                    breed_correct_top5 += 1

n_breed = len(all_breed_true)
print(f"Breed Top-1 Accuracy: {breed_correct_top1}/{n_breed} = {breed_correct_top1/n_breed*100:.2f}%")
print(f"Breed Top-5 Accuracy: {breed_correct_top5}/{n_breed} = {breed_correct_top5/n_breed*100:.2f}%")

---
## 8. Risk Scoring Demo

In [ ]:
# Breed → Risk Score mapping (based on bite statistics & temperament data)
BREED_RISK_TABLE = {
    # High-risk breeds (based on AVMA bite data, insurance claims)
    'pit_bull': 75, 'american_pit_bull_terrier': 75, ' Staffordshire_bull_terrier': 70,
    'rottweiler': 72, 'german_shepherd': 65, 'german_shepherd_dog': 65,
    'doberman_pinscher': 68, 'doberman': 68,
    'chow_chow': 60, 'akita': 58, 'maremma_sheepdog': 55,
    'mastiff': 70, 'bullmastiff': 68, 'english_mastiff': 68,
    'siberian_husky': 55, 'husky': 55,
    'wolfhound': 50, 'irish_wolfhound': 50,
    
    # Moderate breeds
    'boxer': 45, 'dalmatian': 40, 'great_dane': 48,
    'alaskan_malamute': 45, 'samoyed': 35,
    'labrador_retriever': 35, 'labrador': 35,
    'golden_retriever': 25, 'golden_retriever': 25,
    'beagle': 30, 'cocker_spaniel': 28,
    'shar_pei': 42, 'chinese_shar_pei': 42,
    
    # Lower-risk breeds
    'poodle': 20, 'standard_poodle': 22, 'miniature_poodle': 18,
    'chihuahua': 15, 'chihuahua': 15,
    'pomeranian': 12, 'shih_tzu': 10,
    'cavalier_king_charles_spaniel': 15,
    'maltese': 10, 'bichon_frise': 10,
    'french_bulldog': 15, 'french_bulldog': 15,
    'yorkshire_terrier': 12, 'pekingese': 8,
    'toy_poodle': 12, 'papillon': 10,
    
    # Default for unknown breeds
    'unknown': 50,
}

# Emotion → Risk Score mapping
EMOTION_RISK_TABLE = {
    'angry':    90,
    'sad':      30,  # scared/sad dogs can be unpredictable
    'happy':    15,
    'relaxed':  10,
}

# Risk level thresholds
def risk_level(score):
    if score >= 80: return 'SEVERE'
    if score >= 60: return 'HIGH'
    if score >= 40: return 'MODERATE'
    if score >= 20: return 'LOW'
    return 'MINIMAL'

def risk_color(score):
    if score >= 80: return '#DC2626'  # red
    if score >= 60: return '#F97316'  # orange
    if score >= 40: return '#FACC15'  # yellow
    if score >= 20: return '#84CC16'  # yellow-green
    return '#22C55E'  # green

print("Risk tables defined.")

In [ ]:
def calculate_risk(breed_name, breed_conf, emotion_name, emotion_conf, dog_count=1, attacked=False):
    """
    Calculate risk score with full breakdown.
    
    Args:
        breed_name: predicted breed name
        breed_conf: breed prediction confidence (0-1)
        emotion_name: predicted emotion
        emotion_conf: emotion prediction confidence (0-1)
        dog_count: number of dogs observed
        attacked: whether an attack was reported
    
    Returns:
        dict with risk_score, breakdown, and reasoning
    """
    # Breed risk (confidence-weighted)
    raw_breed_risk = BREED_RISK_TABLE.get(breed_name.lower(), 50)
    breed_risk = raw_breed_risk * breed_conf + 50 * (1 - breed_conf)
    
    # Emotion risk (confidence-weighted)
    raw_emotion_risk = EMOTION_RISK_TABLE.get(emotion_name.lower(), 50)
    emotion_risk = raw_emotion_risk * emotion_conf + 50 * (1 - emotion_conf)
    
    # Context modifier
    context_base = 40  # neutral baseline
    pack_bonus = 0
    attack_bonus = 0
    
    if dog_count and dog_count > 1:
        pack_bonus = min(15, (dog_count - 1) * 8)
    if attacked:
        attack_bonus = 20
    
    context_risk = min(100, context_base + pack_bonus + attack_bonus)
    
    # Weighted combination
    score = (
        0.30 * breed_risk +
        0.35 * emotion_risk +
        0.35 * context_risk
    )
    score = round(min(100, max(0, score)))
    
    # Build reasoning
    reasoning_parts = []
    if breed_conf > 0.5:
        reasoning_parts.append(f"Identified as {breed_name} (confidence: {breed_conf:.0%}).")
    else:
        reasoning_parts.append(f"Breed uncertain (best guess: {breed_name}, confidence: {breed_conf:.0%}).")
    
    if emotion_name == 'angry':
        reasoning_parts.append("Dog appears angry/aggressive — multiple threat signals detected.")
    elif emotion_name == 'sad':
        reasoning_parts.append("Dog appears sad/scared — may be unpredictable if cornered.")
    elif emotion_name == 'happy':
        reasoning_parts.append("Dog appears happy/friendly — low threat indicators.")
    else:
        reasoning_parts.append("Dog appears relaxed/calm — no immediate threat signals.")
    
    if pack_bonus > 0:
        reasoning_parts.append(f"Pack of {dog_count} dogs increases risk (+{pack_bonus}).")
    if attack_bonus > 0:
        reasoning_parts.append("Attack reported — significant risk increase.")
    
    return {
        'risk_score': score,
        'risk_level': risk_level(score),
        'risk_color': risk_color(score),
        'breakdown': {
            'breed': {
                'label': breed_name,
                'risk': round(breed_risk, 1),
                'weight': 0.30,
                'contribution': round(0.30 * breed_risk, 1),
                'confidence': round(breed_conf, 3),
            },
            'emotion': {
                'label': emotion_name,
                'risk': round(emotion_risk, 1),
                'weight': 0.35,
                'contribution': round(0.35 * emotion_risk, 1),
                'confidence': round(emotion_conf, 3),
            },
            'context': {
                'risk': round(context_risk, 1),
                'weight': 0.35,
                'contribution': round(0.35 * context_risk, 1),
                'pack_bonus': pack_bonus,
                'attack_bonus': attack_bonus,
            },
        },
        'reasoning': ' '.join(reasoning_parts),
    }

print("Risk calculation function defined.")

In [ ]:
# Demo: Run risk scoring on test images
def denormalize(tensor):
    """Convert normalized tensor back to displayable image."""
    img = tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

# Pick random test images
demo_indices = np.random.choice(len(test_dataset), size=min(8, len(test_dataset)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('SafeDoge Risk Assessment Demo', fontsize=16, fontweight='bold', y=1.02)

model.eval()
with torch.no_grad():
    for idx_pos, demo_idx in enumerate(demo_indices):
        image, true_breed, true_emotion = test_dataset[demo_idx]
        image_tensor = image.unsqueeze(0).to(DEVICE)
        
        breed_logits, emotion_logits = model(image_tensor)
        breed_probs = F.softmax(breed_logits, dim=-1)
        emotion_probs = F.softmax(emotion_logits, dim=-1)
        
        pred_breed_idx = breed_probs[0].argmax().item()
        pred_emotion_idx = emotion_probs[0].argmax().item()
        breed_conf = breed_probs[0][pred_breed_idx].item()
        emotion_conf = emotion_probs[0][pred_emotion_idx].item()
        
        pred_breed = IDX_TO_BREED[pred_breed_idx]
        pred_emotion = IDX_TO_EMOTION[pred_emotion_idx]
        
        # Calculate risk
        risk = calculate_risk(pred_breed, breed_conf, pred_emotion, emotion_conf)
        
        # Display
        ax = axes[idx_pos // 4, idx_pos % 4]
        img = denormalize(image)
        ax.imshow(img)
        
        # Color border by risk
        for spine in ax.spines.values():
            spine.set_edgecolor(risk['risk_color'])
            spine.set_linewidth(4)
        ax.set_xticks([])
        ax.set_yticks([])
        
        # Title with prediction
        true_breed_name = IDX_TO_BREED[true_breed.item()]
        true_emotion_name = IDX_TO_EMOTION[true_emotion.item()]
        
        title_color = risk['risk_color']
        ax.set_title(
            f"Breed: {pred_breed[:18]}\n"
            f"Emotion: {pred_emotion}\n"
            f"Risk: {risk['risk_score']}/100 {risk['risk_level']}",
            fontsize=9, fontweight='bold', color=title_color,
            pad=8,
        )

plt.tight_layout()
plt.savefig(MODEL_DIR / 'risk_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed risk breakdown for one sample
demo_idx = np.random.choice(len(test_dataset))
image, true_breed, true_emotion = test_dataset[demo_idx]

model.eval()
with torch.no_grad():
    image_tensor = image.unsqueeze(0).to(DEVICE)
    breed_logits, emotion_logits = model(image_tensor)
    breed_probs = F.softmax(breed_logits, dim=-1)
    emotion_probs = F.softmax(emotion_logits, dim=-1)
    
    pred_breed_idx = breed_probs[0].argmax().item()
    pred_emotion_idx = emotion_probs[0].argmax().item()
    breed_conf = breed_probs[0][pred_breed_idx].item()
    emotion_conf = emotion_probs[0][pred_emotion_idx].item()
    
    pred_breed = IDX_TO_BREED[pred_breed_idx]
    pred_emotion = IDX_TO_EMOTION[pred_emotion_idx]

risk = calculate_risk(pred_breed, breed_conf, pred_emotion, emotion_conf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Image
axes[0].imshow(denormalize(image))
axes[0].set_title(f"Predicted: {pred_breed} ({breed_conf:.0%}), {pred_emotion} ({emotion_conf:.0%})",
                   fontsize=11, fontweight='bold')
axes[0].axis('off')

# Risk breakdown pie chart
breakdown = risk['breakdown']
labels = [
    f"Breed\n({breakdown['breed']['label'][:12]})",
    f"Emotion\n({breakdown['emotion']['label']})",
    f"Context\n(base)",
]
sizes = [
    breakdown['breed']['contribution'],
    breakdown['emotion']['contribution'],
    breakdown['context']['contribution'],
]
colors = ['#2563EB', '#22C55E', '#F59E0B']

axes[1].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 10})
axes[1].set_title(f"Risk Score: {risk['risk_score']}/100 — {risk['risk_level']}",
                   fontsize=12, fontweight='bold', color=risk['risk_color'])

plt.suptitle('Risk Breakdown', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'risk_breakdown_example.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*60}")
print(f"RISK ASSESSMENT REPORT")
print(f"{'='*60}")
print(f"Breed:      {risk['breakdown']['breed']['label']} ({risk['breakdown']['breed']['confidence']:.0%} confidence)")
print(f"  Risk:     {risk['breakdown']['breed']['risk']} × weight {risk['breakdown']['breed']['weight']} = {risk['breakdown']['breed']['contribution']}")
print(f"Emotion:    {risk['breakdown']['emotion']['label']} ({risk['breakdown']['emotion']['confidence']:.0%} confidence)")
print(f"  Risk:     {risk['breakdown']['emotion']['risk']} × weight {risk['breakdown']['emotion']['weight']} = {risk['breakdown']['emotion']['contribution']}")
print(f"Context:    base=40, pack_bonus={risk['breakdown']['context']['pack_bonus']}, attack_bonus={risk['breakdown']['context']['attack_bonus']}")
print(f"  Risk:     {risk['breakdown']['context']['risk']} × weight {risk['breakdown']['context']['weight']} = {risk['breakdown']['context']['contribution']}")
print(f"{'='*60}")
print(f"FINAL SCORE: {risk['risk_score']}/100 — {risk['risk_level']}")
print(f"REASONING:   {risk['reasoning']}")
print(f"{'='*60}")

---
## 9. Export Model

In [ ]:
# Load best model for export
model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pth', map_location='cpu'))
model.eval()

# 1. Save PyTorch weights (final)
torch.save(model.state_dict(), MODEL_DIR / 'dog_risk_model.pth')
print("Saved: dog_risk_model.pth")

# 2. Export to ONNX
dummy_input = torch.randn(1, 3, 224, 224)
torch.on.export(
    model,
    dummy_input,
    MODEL_DIR / 'dog_risk_model.onnx',
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['breed_logits', 'emotion_logits'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'breed_logits': {0: 'batch_size'},
        'emotion_logits': {0: 'batch_size'},
    },
)
print("Saved: dog_risk_model.onnx")

# 3. Save label mappings
breed_labels = {str(k): v for k, v in IDX_TO_BREED.items()}
with open(MODEL_DIR / 'breed_labels.json', 'w') as f:
    json.dump(breed_labels, f, indent=2)
print("Saved: breed_labels.json")

emotion_labels = {str(k): v for k, v in IDX_TO_EMOTION.items()}
with open(MODEL_DIR / 'emotion_labels.json', 'w') as f:
    json.dump(emotion_labels, f, indent=2)
print("Saved: emotion_labels.json")

# 4. Save breed risk table
with open(MODEL_DIR / 'breed_risk_table.json', 'w') as f:
    json.dump(BREED_RISK_TABLE, f, indent=2)
print("Saved: breed_risk_table.json")

# 5. Save emotion risk table
with open(MODEL_DIR / 'emotion_risk_table.json', 'w') as f:
    json.dump(EMOTION_RISK_TABLE, f, indent=2)
print("Saved: emotion_risk_table.json")

# 6. Save config
with open(MODEL_DIR / 'config.json', 'w') as f:
    json.dump({
        **Config,
        'NUM_BREEDS': NUM_BREEDS,
        'NUM_EMOTIONS': NUM_EMOTIONS,
        'IMAGENET_MEAN': IMAGENET_MEAN,
        'IMAGENET_STD': IMAGENET_STD,
        'breed_to_idx': BREED_TO_IDX,
        'emotion_to_idx': EMOTION_TO_IDX,
    }, f, indent=2)
print("Saved: config.json")

In [ ]:
# List all exported files
print("Exported files:")
for f in sorted(MODEL_DIR.iterdir()):
    size = f.stat().st_size
    if size > 1e6:
        print(f"  {f.name:30s} {size/1e6:.1f} MB")
    else:
        print(f"  {f.name:30s} {size/1e3:.1f} KB")

---
## 10. Inference Function (Ready for Backend Integration)

In [ ]:
import torchvision.transforms as T

# Load everything needed for inference
def load_model(model_path, breed_labels_path, config_path):
    """Load model and metadata for inference."""
    with open(config_path) as f:
        config = json.load(f)
    
    with open(breed_labels_path) as f:
        breed_labels = json.load(f)
    
    num_breeds = len(breed_labels)
    num_emotions = config['NUM_EMOTIONS']
    
    model = MultiTaskDogModel(num_breeds=num_breeds, num_emotions=num_emotions)
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.eval()
    
    transform = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(config['IMAGENET_MEAN'], config['IMAGENET_STD']),
    ])
    
    return model, breed_labels, transform, config


def predict_risk(image_path, model, breed_labels, transform, config):
    """
    Predict breed, emotion, and risk score from a single image.
    
    Args:
        image_path: path to image file
        model: loaded MultiTaskDogModel
        breed_labels: breed index -> name mapping
        transform: image transform pipeline
        config: model configuration
    
    Returns:
        dict with breed, emotion, risk_score, breakdown, reasoning
    """
    IDX_TO_BREED = {int(k): v for k, v in breed_labels.items()}
    IDX_TO_EMOTION = {0: 'angry', 1: 'happy', 2: 'relaxed', 3: 'sad'}
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0)
    
    # Predict
    with torch.no_grad():
        breed_logits, emotion_logits = model(image_tensor)
        breed_probs = F.softmax(breed_logits, dim=-1)
        emotion_probs = F.softmax(emotion_logits, dim=-1)
        
        pred_breed_idx = breed_probs[0].argmax().item()
        pred_emotion_idx = emotion_probs[0].argmax().item()
        breed_conf = breed_probs[0][pred_breed_idx].item()
        emotion_conf = emotion_probs[0][pred_emotion_idx].item()
        
        pred_breed = IDX_TO_BREED[pred_breed_idx]
        pred_emotion = IDX_TO_EMOTION[pred_emotion_idx]
    
    # Calculate risk
    risk = calculate_risk(pred_breed, breed_conf, pred_emotion, emotion_conf)
    
    return {
        'breed': pred_breed,
        'breed_confidence': round(breed_conf, 4),
        'emotion': pred_emotion,
        'emotion_confidence': round(emotion_conf, 4),
        'risk_score': risk['risk_score'],
        'risk_level': risk['risk_level'],
        'risk_color': risk['risk_color'],
        'breakdown': risk['breakdown'],
        'reasoning': risk['reasoning'],
    }

print("Inference function ready.")

In [ ]:
# Test the inference function
loaded_model, loaded_breed_labels, loaded_transform, loaded_config = load_model(
    MODEL_DIR / 'dog_risk_model.pth',
    MODEL_DIR / 'breed_labels.json',
    MODEL_DIR / 'config.json',
)

# Run on a test image
test_img_path = test_dataset.image_paths[0]
result = predict_risk(test_img_path, loaded_model, loaded_breed_labels, loaded_transform, loaded_config)

print(f"Image: {test_img_path}")
print(json.dumps(result, indent=2))

In [ ]:
# Batch inference example
def predict_batch_risk(image_paths, model, breed_labels, transform, config):
    """Predict risk for multiple images."""
    results = []
    for path in image_paths:
        try:
            result = predict_risk(path, model, breed_labels, transform, config)
            result['image_path'] = str(path)
            results.append(result)
        except Exception as e:
            results.append({'image_path': str(path), 'error': str(e)})
    return results

# Demo batch prediction
sample_paths = test_dataset.image_paths[:5]
batch_results = predict_batch_risk(sample_paths, loaded_model, loaded_breed_labels, loaded_transform, loaded_config)

for r in batch_results:
    if 'error' in r:
        print(f"ERROR: {r['image_path']}: {r['error']}")
    else:
        print(f"{r['breed']:25s} | {r['emotion']:10s} | Risk: {r['risk_score']:3d}/100 {r['risk_level']:10s} | Conf: B={r['breed_confidence']:.0%} E={r['emotion_confidence']:.0%}")

---
## Summary

### Exported Files
| File | Purpose |
|------|--------|
| `dog_risk_model.pth` | PyTorch model weights |
| `dog_risk_model.onnx` | ONNX export (for non-PyTorch deployment) |
| `breed_labels.json` | Class index → breed name mapping |
| `emotion_labels.json` | Class index → emotion name mapping |
| `breed_risk_table.json` | Breed → inherent risk score |
| `emotion_risk_table.json` | Emotion → risk score |
| `config.json` | Model config + label mappings |

### Risk Formula
```
score = 0.30 × breed_risk + 0.35 × emotion_risk + 0.35 × context
```

### Usage in Backend
```python
from safe_doge_model import load_model, predict_risk

model, breed_labels, transform, config = load_model(
    'models/dog_risk_model.pth',
    'models/breed_labels.json',
    'models/config.json',
)

result = predict_risk('photo.jpg', model, breed_labels, transform, config)
print(result['risk_score'])  # 0-100
print(result['reasoning'])   # explanation
```